# 59_sliding_window_attn

Audience: junior researcher

- Challenge path: `challenges/hard/59_sliding_window_attn`
- Source spec: [challenges/hard/59_sliding_window_attn/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/59_sliding_window_attn/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement <strong>Sliding Window Self-Attention</strong> for a given set of matrices.
  Before introducing the sliding window version, let's first recall standard Self-Attention.
</p>

<h3>1. Standard Softmax Attention</h3>
<p>
  Given query matrix <code>Q</code>, key matrix <code>K</code>, and value matrix <code>V</code>, each position <code>i</code> attends to all positions <code>j</code> using a softmax-weighted sum:
</p>

<p style="text-align:center;">
  \( \text{score}_{i,j} = \frac{Q_i \cdot K_j}{\sqrt{d}} \)
</p>

<p style="text-align:center;">
  \( \text{output}_i = \sum_{j=1}^{M} \text{softmax}(\text{score}_{i,*})_j \cdot V_j \)
</p>

<p>
  In other words, each query computes similarity with all keys, applies a softmax to get attention weights, and then computes a weighted sum of values.
</p>

<h3>2. Sliding Window Self-Attention</h3>
<p>
  Sliding Window Attention modifies standard attention by restricting each query to attend only to a local window around its position.
</p>

<ul>
  <li>For each position <code>i</code>, only consider the keys and values within a window of size <code>window_size</code> around <code>i</code> (positions <code>[i-window_size, ..., i+window_size]</code>).</li>
  <li>Compute similarity scores between <code>Q<sub>i</sub></code> and the keys in this window:</li>
</ul>

<p style="text-align:center;">
  \( \text{score}_{i,j} = \frac{Q_i \cdot K_j}{\sqrt{d}} \)
</p>

<ul>
  <li>Apply <code>softmax</code> over these local scores to obtain attention weights.</li>
  <li>Use the weights to compute a weighted average of the values in the same window:</li>
</ul>

<p style="text-align:center;">
  \( \text{output}_i = \sum_{j \in [i-\text{window_size}, \, i+\text{window_size}]} \text{softmax}(\text{score}_{i,*})_j \cdot V_j \)
</p>

<p>
  In short, each query only attends to its nearby neighbors.
</p>

<svg width="320" height="320" viewBox="0 0 320 320" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;">
  <style>
    text { font-family: monospace; fill: #cccccc; }
  </style>
  <!-- Background -->
  <rect width="320" height="320" fill="#1a1a1a" rx="6"/>

  <!-- Axis labels -->
  <text x="185" y="16" text-anchor="middle" font-size="11">key position &#x2192;</text>
  <text x="14" y="168" text-anchor="middle" font-size="11" transform="rotate(-90,14,168)">query position</text>

  <!-- Column headers (positions 0-7) -->
  <g font-size="10" text-anchor="middle">
    <text x="62" y="36">0</text>
    <text x="93" y="36">1</text>
    <text x="124" y="36">2</text>
    <text x="155" y="36">3</text>
    <text x="186" y="36">4</text>
    <text x="217" y="36">5</text>
    <text x="248" y="36">6</text>
    <text x="279" y="36">7</text>
  </g>

  <!-- Row headers (positions 0-7) -->
  <g font-size="10" text-anchor="middle">
    <text x="34" y="58">0</text>
    <text x="34" y="89">1</text>
    <text x="34" y="120">2</text>
    <text x="34" y="151">3</text>
    <text x="34" y="182">4</text>
    <text x="34" y="213">5</text>
    <text x="34" y="244">6</text>
    <text x="34" y="275">7</text>
  </g>

  <!-- 8x8 grid: cell is blue (#2a4a7a) if |row-col| <= 2, else dark (#2a2a2a) -->
  <!-- Row 0: cols 0,1,2 blue -->
  <rect x="46" y="42" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="42" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="42" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 1: cols 0,1,2,3 blue -->
  <rect x="46" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 2: cols 0,1,2,3,4 blue -->
  <rect x="46" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="104" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="104" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="104" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 3: cols 1,2,3,4,5 blue -->
  <rect x="46" y="135" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="135" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="135" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 4: cols 2,3,4,5,6 blue -->
  <rect x="46" y="166" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="166" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="166" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 5: cols 3,4,5,6,7 blue -->
  <rect x="46" y="197" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="197" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="197" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 6: cols 4,5,6,7 blue -->
  <rect x="46" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 7: cols 5,6,7 blue -->
  <rect x="46" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="259" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="259" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="259" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>

  <!-- Grid border -->
  <rect x="46" y="42" width="248" height="248" fill="none" stroke="#555555" stroke-width="1"/>

  <!-- Label: window_size = 2 -->
  <text x="170" y="308" text-anchor="middle" font-size="12" fill="#999999">window_size = 2</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The
    <code>solve</code> function signature must remain unchanged
  </li>
  <li>The final result must be stored in the output matrix
    <code>output</code>
  </li>
</ul>
<h2>Example 1:</h2>
<p>
<strong>Input:</strong><br>
<code>Q</code> (2×4):
\[
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
\]
<code>K</code> (2×4):
\[
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
\]
<code>V</code> (2×4):
\[
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0
\end{bmatrix}
\]
<code>window_size</code>: 1
</p>

<p>
<strong>Output:</strong><br>
<code>output</code> (2×4):
\[
\begin{bmatrix}
2.5101628 & 3.5101628 & 4.510163 & 5.510163 \\
3.4898374 & 4.4898376 & 5.4898376 & 6.489837
\end{bmatrix}
\]
</p>


<h2>Example 2:</h2>
<p>
  <strong>Input:</strong><br>
  <code>Q</code> (2×3):
  \[
  \begin{bmatrix}
  0.0 & 0.0 & 0.0 \\
  0.0 & 1.0 & 0.0
  \end{bmatrix}
  \]
  <code>K</code> (2×3):
  \[
  \begin{bmatrix}
  1.0 & 0.0 & 0.0 \\
  0.0 & 1.0 & 0.0
  \end{bmatrix}
  \]
  <code>V</code> (2×3):
  \[
  \begin{bmatrix}
  1.0 & 2.0 & 3.0 \\
  5.0 & 6.0 & 7.0
  \end{bmatrix}
  \]
  <code>window_size</code>: 1
  </p>

  <p>
  <strong>Output:</strong><br>
  <code>output</code> (2×3):
  \[
  \begin{bmatrix}
  3.0 & 4.0 & 5.0 \\
  3.5618298 & 4.56183 & 5.5618296
  \end{bmatrix}
  \]
  </p>



<h2>Constraints</h2>
<ul>
  <li>Matrix <code>Q</code>, <code>K</code>, and <code>V</code> are all of size <code>M×d</code></li>
  <li>1 &le; <code>M</code> &le; 10000</li>
  <li>1 &le; <code>d</code> &le; 128</li>
  <li>1 &le; <code>window_size</code> &le; 32</li>
  <li>All elements in <code>Q</code>, <code>K</code>, and <code>V</code> are sampled from<code>[-100.0, 100.0]</code></li>
  <li>Data type for all matrices is <code>float32</code></li>

  <li>Performance is measured with <code>M</code> = 5,000</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/59_sliding_window_attn/solution/solution.pytorch.py`

In [ ]:
import math

import torch


def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    M: int,
    d: int,
    window_size: int,
):
    scores = torch.matmul(Q, K.t()) / math.sqrt(d)
    idxs = torch.arange(M, device=Q.device)
    mask = (idxs[None, :] - idxs[:, None]).abs() > window_size
    scores = scores.masked_fill(mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    torch.matmul(attn, V, out=output)


## Jax

Source: `challenges/hard/59_sliding_window_attn/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

from typing import Any
import math

import torch

try:
    import jax
    import jax.numpy as jnp
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None


def _torch_impl(Q: Any, K: Any, V: Any, M: int, d: int, window_size: int):
    scores = torch.matmul(Q, K.t()) / math.sqrt(d)
    idxs = torch.arange(M, device=Q.device)
    mask = (idxs[None, :] - idxs[:, None]).abs() > window_size
    scores = scores.masked_fill(mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V


def _jax_impl(Q: Any, K: Any, V: Any, M: int, d: int, window_size: int):
    scores = jnp.matmul(Q, K.T) / math.sqrt(d)
    idxs = jnp.arange(M)
    mask = jnp.abs(idxs[None, :] - idxs[:, None]) > window_size
    scores = jnp.where(mask, -jnp.inf, scores)
    scores = scores - jnp.max(scores, axis=-1, keepdims=True)
    weights = jnp.exp(scores)
    weights = weights / jnp.sum(weights, axis=-1, keepdims=True)
    return jnp.matmul(weights, V)


if jax is None:

    def solve(Q: Any, K: Any, V: Any, M: int, d: int, window_size: int):
        return _torch_impl(Q, K, V, M, d, window_size)

else:
    solve = jax.jit(_jax_impl, static_argnames=("M", "d", "window_size"))


## Triton

Source: `challenges/hard/59_sliding_window_attn/solution/solution.triton.py`

In [ ]:
import math

import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    M: int,
    d: int,
    window_size: int,
):
    scores = torch.matmul(Q, K.t()) / math.sqrt(d)
    idxs = torch.arange(M, device=Q.device)
    mask = (idxs[None, :] - idxs[:, None]).abs() > window_size
    scores = scores.masked_fill(mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    torch.matmul(attn, V, out=output)


## Mlx

Source: `challenges/hard/59_sliding_window_attn/solution/solution.mlx.py`

In [ ]:
from __future__ import annotations

from typing import Any
import importlib.util
import math

import torch


def _torch_impl(Q: Any, K: Any, V: Any, output: Any, M: int, d: int, window_size: int):
    scores = torch.matmul(Q, K.t()) / math.sqrt(d)
    idxs = torch.arange(M, device=Q.device)
    mask = (idxs[None, :] - idxs[:, None]).abs() > window_size
    scores = scores.masked_fill(mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    output.copy_(attn @ V)
    return output


def solve(Q: Any, K: Any, V: Any, output: Any, M: int, d: int, window_size: int):
    if importlib.util.find_spec("mlx.core") is None:
        return _torch_impl(Q, K, V, output, M, d, window_size)

    import mlx.core as mx

    scores = (Q @ K.T) / math.sqrt(d)
    idxs = mx.arange(M)
    mask = mx.abs(idxs[None, :] - idxs[:, None]) > window_size
    scores = mx.where(mask, -mx.inf, scores)
    scores = scores - mx.max(scores, axis=-1, keepdims=True)
    weights = mx.exp(scores)
    weights = weights / mx.sum(weights, axis=-1, keepdims=True)
    result = weights @ V
    output[...] = result
    return output


## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.